# GC Column RI Coverage Tool

Paste a list of volatile compound names and this notebook will check NIST WebBook for retention index (RI) data on both polar and non-polar GC phases, then rank which phases and phase-**families** (Wax, FFAP, DB-5-type, etc.) have the best literature coverage for your compound list.

**No installation needed** - this runs entirely in your browser via Google Colab.

### First-time setup (do this once per session)
1. In the menu above, go to **Runtime > Change runtime type**.
2. Under "Runtime type," select **R**. Click Save.
   *(The "R" indicator in the bottom-right corner only appears once you start running cells - it's normal not to see it beforehand.)*
3. Run the cells below in order (click the play button on each, or Shift+Enter).

### Important limitation
Colab does **not** save installed packages between sessions. If you come back to this notebook later (or after it's been idle a while), you'll need to re-run the installation cell below again - it takes a minute or two.

### Choosing your RI type(s)
NIST reports retention index under up to four different calculation methods, which are **not interchangeable** - pick based on your own GC method: `kovats` (isothermal runs), `linear` (Van den Dool & Kratz, for temperature-programmed runs - most common in modern food/flavor work), `alkane`, and `lee` (PAH-specific, unlikely to matter for aroma volatiles). The default fetches all four, which is safest but roughly **4x slower** than fetching just one - narrow `RI_TYPES` in the code below to only what you actually use if you want a faster run.

### A note on speed and occasional errors
This notebook deliberately paces its requests to NIST's server (small delays plus automatic retries) rather than firing everything at once. It also resolves each compound to a CAS number first (via a direct PubChem API call) before querying NIST, since NIST's own name search can land on the wrong record when a compound has more than one database entry under the same name - a confirmed issue for several common compounds (e.g. limonene, alpha-pinene, methanethiol). If CAS resolution fails or comes up empty, it automatically falls back to a name-based search. When everything still fails, the script checks a shared, moderated list of known problem compounds and gives you a direct link to check manually.

### Known-issues pointer sheet
Failed compounds are automatically checked against a shared Google Sheet of known problem compounds and their direct NIST WebBook links. This sheet stores **pointers only - never actual RI data** - NIST's Standard Reference Data has its own copyright terms restricting reproduction of the underlying data itself, so nothing from NIST's database is ever copied or redistributed through this mechanism.

The sheet is moderated: new submissions come in via a Google Form, and a moderator reviews and adds validated entries to the published sheet the tool actually reads.

In [ ]:
# Run this once per session. Takes 1-2 minutes.
install.packages(c("webchem", "dplyr", "writexl"))


### Paste your compound list and choose your RI type(s)
Edit the block below - one compound name per line, plus RI_TYPES if you want a narrower/faster run - then run this cell.

In [ ]:
# ---- 0. Paste your compound list here (one per line) --------------
MY_COMPOUNDS_RAW <- "
hexanal
limonene
alpha-pinene
1-octen-3-ol
nonanal
"

# ---- 0b. Which retention index type(s) do you need? -----------------
# NIST WebBook reports RI under up to four different calculation methods,
# which are NOT interchangeable - pick based on your own GC method:
#   "kovats" - Kovats RI, for ISOTHERMAL runs
#   "linear" - Van den Dool & Kratz RI, for TEMPERATURE-PROGRAMMED runs
#              (most common in modern food/flavor GC-O work)
#   "alkane" - Normal Alkane RI (a simpler/older related index)
#   "lee"    - Lee RI (developed specifically for PAH analysis, unlikely
#              to be relevant for aroma volatiles, included for completeness)
# Leave as all four (the default) to fetch everything - safest, but
# slowest. Narrowing this to just what you actually use (e.g. just
# "linear") meaningfully speeds up the script, since only the selected
# type(s) are queried from NIST at all - not fetched and filtered after.
RI_TYPES <- c("kovats", "linear", "alkane", "lee")

# install.packages(c("webchem", "dplyr", "writexl"))  # uncomment on first run


In [ ]:
library(webchem)
library(dplyr)
library(writexl)

my_compounds <- trimws(strsplit(MY_COMPOUNDS_RAW, "\n")[[1]])
my_compounds <- unique(my_compounds[my_compounds != ""])
n_compounds  <- length(my_compounds)
cat("Loaded", n_compounds, "unique compound names.\n\n")

# ---- 1. Greek-letter normalization ---------------------------------
# NIST sometimes returns names using actual Unicode Greek letters
# (e.g. "\u03b1" = alpha) or italic mathematical Greek variants
# (e.g. "\U0001D6FF", a DIFFERENT character that just looks like delta).
# Both break Excel's default CSV encoding and won't match a
# spelled-out query like "alpha-pinene". This maps both variants to
# plain spelled-out form.
clean_greek <- function(x) {
  greek_map <- c(
    "\u03b1" = "alpha", "\u0391" = "alpha",
    "\u03b2" = "beta",  "\u0392" = "beta",
    "\u03b3" = "gamma", "\u0393" = "gamma",
    "\u03b4" = "delta", "\u0394" = "delta",
    "\u03b5" = "epsilon","\u0395" = "epsilon",
    "\u03c9" = "omega", "\u03a9" = "omega",
    "\U0001D6FC" = "alpha", "\U0001D6FD" = "beta",
    "\U0001D6FE" = "gamma", "\U0001D6FF" = "delta",
    "\U0001D700" = "epsilon", "\U0001D714" = "omega"
  )
  for (ch in names(greek_map)) x <- gsub(ch, greek_map[[ch]], x, fixed = TRUE)
  x
}

# ---- 2a. Resolve compound name to CAS number via PubChem (direct API) --
# NIST's own name search can land on a "Search Results" disambiguation page
# when a compound has more than one database entry under the same name
# (confirmed for limonene, alpha-pinene, and methanethiol - all have
# duplicate NIST entries). A CAS number routes directly and unambiguously
# to a single record, sidestepping that failure mode entirely.
#
# Two earlier approaches were tried and ruled out: the Chemical Translation
# Service (CTS) was confirmed down ("Service not available"), and webchem's
# own pc_synonyms() was confirmed to silently return an incomplete synonym
# list (just one computed name, no CAS) even though PubChem's raw API
# returns the full list correctly for the same compound. So this calls
# PubChem's PUG REST API directly via httr, bypassing that wrapper bug:
# name -> CID, then CID -> full synonym list, then pattern-match for a
# CAS-shaped string within it. This is a solid heuristic, not a guarantee,
# for compounds with multiple registered CAS numbers (rare) - if wrong or
# unavailable, the name-based fallback below still applies as a safety net.
resolve_cas <- function(name) {
  base <- "https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound"

  cid <- tryCatch({
    url <- paste0(base, "/name/", utils::URLencode(name, reserved = TRUE), "/cids/JSON")
    resp <- httr::GET(url)
    if (httr::status_code(resp) != 200) return(NA)
    parsed <- httr::content(resp, as = "parsed", type = "application/json")
    as.character(parsed$IdentifierList$CID[[1]])
  }, error = function(e) NA)
  if (is.na(cid)) return(NA)

  cas <- tryCatch({
    url <- paste0(base, "/cid/", cid, "/synonyms/JSON")
    resp <- httr::GET(url)
    if (httr::status_code(resp) != 200) return(NA)
    parsed <- httr::content(resp, as = "parsed", type = "application/json")
    syns <- unlist(parsed$InformationList$Information[[1]]$Synonym)
    cas_candidates <- syns[grepl("^[0-9]{2,7}-[0-9]{2}-[0-9]$", syns)]
    if (length(cas_candidates) == 0) return(NA)
    cas_candidates[1]
  }, error = function(e) NA)
  cas
}

# ---- 2b. Query NIST one compound at a time, CAS-first with name fallback --
# Tries the resolved CAS number first (unambiguous, direct hit). Only if
# that fails (CAS resolution failed, or NIST has nothing under that CAS)
# does it fall back to the older name-based search, which remains useful
# for compounds PubChem doesn't recognize but NIST's name search handles
# fine on its own. A small delay and outer retry-with-backoff loop also
# help with transient 504 Gateway Timeout errors. CAS resolution happens
# ONCE per compound (not once per RI type) since it doesn't depend on type.
query_with_progress <- function(names_vec, polarity_val, ri_types, max_attempts = 3, pace_seconds = 1) {
  n <- length(names_vec)
  results <- list()

  query_once <- function(query_val, from_type, ri_type_val) {
    out <- NULL
    for (attempt in seq_len(max_attempts)) {
      out <- tryCatch(
        nist_ri(query_val, from = from_type, type = ri_type_val, polarity = polarity_val),
        error = function(e) {
          cat("   -> error (attempt", attempt, "of", max_attempts, "):", conditionMessage(e), "\n")
          NULL
        }
      )
      got_real_data <- !is.null(out) && "RI" %in% names(out) && any(!is.na(out$RI))
      if (got_real_data) break
      if (attempt < max_attempts) {
        wait <- pace_seconds * attempt * 2
        cat("   retrying in", wait, "seconds...\n")
        Sys.sleep(wait)
      }
    }
    out
  }

  for (i in seq_along(names_vec)) {
    compound_name <- names_vec[i]
    cas <- resolve_cas(compound_name)
    if (!is.na(cas)) cat("  ", compound_name, "- resolved CAS:", cas, "\n")

    for (ri_type_val in ri_types) {
      cat(sprintf("[%s/%s] %d/%d: %s\n", polarity_val, ri_type_val, i, n, compound_name))

      res <- NULL
      if (!is.na(cas)) {
        res <- query_once(cas, "cas", ri_type_val)
      }

      got_real_data <- !is.null(res) && "RI" %in% names(res) && any(!is.na(res$RI))
      if (!got_real_data) {
        res <- query_once(compound_name, "name", ri_type_val)
      }

      if (!is.null(res)) {
        res$query <- compound_name  # normalize back to the original name so
                                     # downstream matching (family/coverage/
                                     # known-issues) stays consistent regardless
                                     # of whether CAS or name search succeeded
        res$ri_type <- ri_type_val
        results[[length(results) + 1]] <- res
      }
      Sys.sleep(pace_seconds)  # be gentle on NIST's server between requests
    }
  }
  dplyr::bind_rows(results)
}

cat("--- Querying non-polar phases ---\n")
nonpolar_ri <- query_with_progress(my_compounds, "non-polar", RI_TYPES)
cat("\n--- Querying polar phases ---\n")
polar_ri <- query_with_progress(my_compounds, "polar", RI_TYPES)

nonpolar_ri <- nonpolar_ri %>% mutate(across(where(is.character), clean_greek))
polar_ri    <- polar_ri    %>% mutate(across(where(is.character), clean_greek))

# ---- 3. Phase -> family lookup table -------------------------------
# Curated from manufacturer documentation. NOT exhaustive.
# Case-insensitive matching is applied below.
phase_family_table <- tribble(
  ~phase,                 ~family,
  # --- Wax / plain PEG family ---
  "RTX-Wax",              "Wax",
  "BP-20",                "Wax",
  "DB-Wax",               "Wax",
  "DB-Wax Etr",           "Wax",
  "DB-WAXetr",            "Wax",
  "DB-WAXms",             "Wax",
  "HP-20M",               "Wax",
  "HP-Innowax",           "Wax",
  "Innowax",              "Wax",
  "Supelcowax-10",        "Wax",
  "Supelcowax",           "Wax",
  "Polyethylene Glycol",  "Wax",
  "PEG",                  "Wax",
  "PEG-20M",              "Wax",
  "HP-Wax",               "Wax",
  "HP20Wax",              "Wax",
  "Carbowax 20M",         "Wax",
  "Carbowax",             "Wax",
  "CP-Wax 52CB",          "Wax",
  "CP-WAX 57CB",          "Wax",
  "AT-Wax",               "Wax",
  "Stabilwax",            "Wax",
  "Stabilwax-MS",         "Wax",
  "VF-WAXms",             "Wax",
  "ZB-Wax",               "Wax",
  "EC-WAX",               "Wax",
  "SOLGel-Wax",           "Wax",
  "Megawax",              "Wax",  # NOT independently verified - naming-convention guess only, please confirm
  # --- FFAP / acid-modified PEG family (includes known naming traps) ---
  "DB-FFAP",              "FFAP",
  "HP-FFAP",              "FFAP",
  "FFAP",                 "FFAP",
  "SP-1000",              "FFAP",   # naming trap: no "FFAP" in name
  "SPB-1000",             "FFAP",
  "OV-351",               "FFAP",
  "CP-Wax 58CB",          "FFAP",   # naming trap: looks like a Wax phase
  "CP-Wax 58 FFAP CB",    "FFAP",
  "Nukol",                "FFAP",
  "Stabilwax-DA",         "FFAP",
  "BP-21",                "FFAP",
  "TRB-FFAP",             "FFAP",
  "007-FFAP",             "FFAP",
  # --- Non-polar: 100% dimethylpolysiloxane ---
  "DB-1",                 "Non-polar (100% dimethyl)",
  "DB-1ms",               "Non-polar (100% dimethyl)",
  "HP-1",                 "Non-polar (100% dimethyl)",
  "HP-1ms",               "Non-polar (100% dimethyl)",
  "HP-101",               "Non-polar (100% dimethyl)",
  "OV-1",                 "Non-polar (100% dimethyl)",
  "OV-101",               "Non-polar (100% dimethyl)",
  "SE-30",                "Non-polar (100% dimethyl)",
  "Rtx-1",                "Non-polar (100% dimethyl)",
  "RTX-1",                "Non-polar (100% dimethyl)",
  "Rtx-1ms",              "Non-polar (100% dimethyl)",
  "Methyl Silicone",      "Non-polar (100% dimethyl)",
  "Apiezon M",            "Non-polar (100% dimethyl)",  # hydrocarbon grease phase, not independently verified
  "CP Sil 5 CB",          "Non-polar (100% dimethyl)",  # naming trap: NOT same family as CP-Sil 8 CB
  "Petrocol DH",          "Non-polar (100% dimethyl)",  # not independently verified
  "RSL-200",              "Non-polar (100% dimethyl)",  # not independently verified
  "BP-1",                 "Non-polar (100% dimethyl)",
  "SPB-1",                "Non-polar (100% dimethyl)",
  "Ultra-1",              "Non-polar (100% dimethyl)",
  "CP-Sil PONA GB",       "Non-polar (100% dimethyl)",
  "CP-Sil PONA CB",       "Non-polar (100% dimethyl)",
  "SPB-Sulfur",           "Non-polar (100% dimethyl)",  # confirmed: thick-film SPB-1, same 100% dimethylpolysiloxane chemistry
  # --- Low-polarity: 5%-phenyl ("DB-5 type") ---
  "5 % Phenyl methyl siloxane", "Low-polarity (5%-phenyl)",
  "DB-5",                 "Low-polarity (5%-phenyl)",
  "DB-5ms",               "Low-polarity (5%-phenyl)",
  "DB-5MS",               "Low-polarity (5%-phenyl)",
  "HP-5",                 "Low-polarity (5%-phenyl)",
  "HP-5ms",               "Low-polarity (5%-phenyl)",
  "HP-5MS",               "Low-polarity (5%-phenyl)",
  "Rtx-5",                "Low-polarity (5%-phenyl)",
  "RTX-5",                "Low-polarity (5%-phenyl)",
  "RTX-5 MS",             "Low-polarity (5%-phenyl)",
  "SPB-5",                "Low-polarity (5%-phenyl)",
  "CP-Sil 8 CB",          "Low-polarity (5%-phenyl)",   # naming trap: NOT same family as CP-Sil 5 CB
  "CP Sil 8 CB",          "Low-polarity (5%-phenyl)",
  "CP-SIL8",              "Low-polarity (5%-phenyl)",
  "CP-Sil8",              "Low-polarity (5%-phenyl)",
  "CP-Sil 8CB-MS",        "Low-polarity (5%-phenyl)",
  "SE-54",                "Low-polarity (5%-phenyl)",
  "SE-52",                "Low-polarity (5%-phenyl)",
  "VF-5ms",               "Low-polarity (5%-phenyl)",
  "VF-5MS",               "Low-polarity (5%-phenyl)",
  "ZB-5",                 "Low-polarity (5%-phenyl)",
  "BPX5",                 "Low-polarity (5%-phenyl)",
  "BPX-5",                "Low-polarity (5%-phenyl)",
  "BP-5",                 "Low-polarity (5%-phenyl)",
  "Ultra-2",              "Low-polarity (5%-phenyl)",
  "SLB-5MS",              "Low-polarity (5%-phenyl)",
  "SLB-5ms",              "Low-polarity (5%-phenyl)",
  "Equity-5MS",           "Low-polarity (5%-phenyl)",
  "MDN-5",                "Low-polarity (5%-phenyl)",   # not independently verified
  "Mega 5MS",             "Low-polarity (5%-phenyl)",   # not independently verified
  "Optima 5",             "Low-polarity (5%-phenyl)",   # not independently verified
  "RH-5MS",               "Low-polarity (5%-phenyl)",   # not independently verified
  # --- Mid-polarity: 50% phenyl ---
  "DB-17",                "Mid-polarity (50%-phenyl)",
  "DB-17ms",              "Mid-polarity (50%-phenyl)",
  "Rtx-50",               "Mid-polarity (50%-phenyl)",
  # --- Mid-polarity: 14% cyanopropylphenyl ---
  "DB-1701",              "Mid-polarity (14%-cyanopropylphenyl)",
  "Rtx-1701",             "Mid-polarity (14%-cyanopropylphenyl)",
  "OV-1701",              "Mid-polarity (14%-cyanopropylphenyl)",
  "CP-Sil 19 CB",         "Mid-polarity (14%-cyanopropylphenyl)",
  # --- Highly polar cyanopropyl (mainly FAME analysis) ---
  "DB-23",                "Highly polar (cyanopropyl)",
  "CP-Sil 88",            "Highly polar (cyanopropyl)",
  "SP-2380",              "Highly polar (cyanopropyl)",
  "BPX70",                "Highly polar (cyanopropyl)",
  "Omegawax",             "Highly polar (cyanopropyl)",
  "SP-2560",              "Highly polar (cyanopropyl)",
  # --- No match ---
  "NA",                   "No match"
)

classify_family <- function(phase_vec) {
  key        <- tolower(trimws(phase_vec))
  lookup_key <- tolower(trimws(phase_family_table$phase))
  idx        <- match(key, lookup_key)
  family     <- phase_family_table$family[idx]
  family[is.na(family) & key == "na"] <- "No match"
  family[is.na(family) & key != "na"] <- "UNCLASSIFIED - add to lookup table"
  family
}

nonpolar_ri <- nonpolar_ri %>% mutate(family = classify_family(phase))
polar_ri    <- polar_ri    %>% mutate(family = classify_family(phase))

# ---- 4. Coverage summaries ------------------------------------------
# Distinct-compound counts (NOT raw row counts - a compound can have
# many literature RI records on the same phase, which would otherwise
# overcount). This replicates the Excel "Distinct Count" pivot table.
summarize_by <- function(df, group_col, total_n) {
  df %>%
    filter(!is.na(RI)) %>%
    group_by(across(all_of(group_col))) %>%
    summarise(distinct_compounds = n_distinct(query), .groups = "drop") %>%
    mutate(coverage_pct = round(100 * distinct_compounds / total_n, 1)) %>%
    arrange(desc(distinct_compounds))
}

# ---- 4b. RI summary statistics (mean/SD across literature records) --
# A single compound-phase combination can have dozens of published RI
# records from different studies/instruments. Rather than making you
# scroll through all of them, this collapses each (compound, group,
# RI type) combination down to count/mean/SD/min/max - a quick way to
# judge both a representative value AND how much agreement exists across
# the literature (a tight SD suggests a reliable value; a wide SD may
# flag isomer confusion or genuine measurement variability). RI TYPE is
# always kept as a separate grouping level - averaging Kovats and Van den
# Dool/Kratz values together would not be chemically meaningful, since
# they're calculated differently (isothermal vs temperature-programmed).
summarize_ri_stats <- function(df, group_col) {
  df %>%
    filter(!is.na(RI)) %>%
    group_by(across(all_of(c("query", group_col, "ri_type")))) %>%
    summarise(
      n_records = n(),
      mean_RI = round(mean(RI, na.rm = TRUE), 1),
      sd_RI   = round(sd(RI, na.rm = TRUE), 1),
      min_RI  = round(min(RI, na.rm = TRUE), 1),
      max_RI  = round(max(RI, na.rm = TRUE), 1),
      .groups = "drop"
    ) %>%
    arrange(query, ri_type, across(all_of(group_col)))
}

# ---- 3b. Check known-issues pointer sheet for failed compounds -------
# A small number of compounds can fail automated NIST lookup even though
# they load fine in a real browser - e.g. if a compound has duplicate
# NIST entries under the same name, the search can land on a
# disambiguation page instead of the actual record. (Limonene and
# alpha-pinene were early cases of this, but the CAS-first resolution
# above - see 2a - fixed lookup for those two, so they're no longer
# expected to fail; this section exists for whatever fails next.)
# Rather than silently returning NA, this checks a shared, moderator-
# maintained Google Sheet of KNOWN problem compounds and their direct
# NIST WebBook URLs - so instead of a dead end, you get a one-click link
# to check the value yourself.
#
# IMPORTANT: this sheet stores POINTERS (compound name, CAS number, NIST
# URL) only - never actual RI data. NIST's Standard Reference Data has
# its own copyright terms restricting reproduction/redistribution of the
# underlying data itself; a shared pointer list avoids that issue since
# nothing from NIST's database is being copied or stored here.
#
# To use this: publish a Google Sheet to the web as CSV (File > Share >
# Publish to web > CSV) with columns: compound, cas_number, nist_url,
# notes. Only trusted moderators should have EDIT access to the sheet
# itself - the published CSV link can be safely public/read-only.
# Leave KNOWN_ISSUES_SHEET_URL blank to skip this step entirely.
# IMPORTANT: the URL must be wrapped in quotes, like this:
#   KNOWN_ISSUES_SHEET_URL <- "https://docs.google.com/spreadsheets/d/e/.../pub?output=csv"
# This must be the PUBLISHED-AS-CSV link of the actual data sheet (File >
# Share > Publish to web > CSV) - NOT a Google Form link, and not the
# normal "edit" or "view" link to the sheet.
KNOWN_ISSUES_SHEET_URL <- "https://docs.google.com/spreadsheets/d/e/2PACX-1vS325JRlaCd1VplDI8vN-ogKrLwr_NFRiCiwvXRFBSgofcw2uT9AAZSYHx1xCqje4JyTs6Ovm-BxY9g/pub?output=csv"

check_known_issues <- function(failed_compounds, sheet_url) {
  if (sheet_url == "" || length(failed_compounds) == 0) return(NULL)
  pointer_sheet <- tryCatch(
    read.csv(sheet_url, stringsAsFactors = FALSE),
    error = function(e) {
      cat("Could not fetch known-issues sheet:", conditionMessage(e), "\n")
      NULL
    }
  )
  if (is.null(pointer_sheet)) return(NULL)

  # Normalize column names to lowercase so header casing (e.g. "Compound"
  # vs "compound") doesn't cause a silent failure to match.
  names(pointer_sheet) <- tolower(trimws(names(pointer_sheet)))
  required_cols <- c("compound", "cas_number", "nist_url", "notes")
  missing_cols <- setdiff(required_cols, names(pointer_sheet))
  if (length(missing_cols) > 0) {
    cat("Known-issues sheet is missing expected column(s):", paste(missing_cols, collapse = ", "), "\n")
    return(NULL)
  }

  key <- tolower(trimws(failed_compounds))
  sheet_key <- tolower(trimws(pointer_sheet$compound))
  idx <- match(key, sheet_key)

  data.frame(
    compound  = failed_compounds,
    known_issue = !is.na(idx),
    cas_number  = ifelse(is.na(idx), NA, pointer_sheet$cas_number[idx]),
    nist_url    = ifelse(is.na(idx), NA, pointer_sheet$nist_url[idx]),
    notes       = ifelse(is.na(idx), NA, pointer_sheet$notes[idx]),
    stringsAsFactors = FALSE
  )
}

# A compound "failed" if it has no real RI data on EITHER polarity
failed_nonpolar <- my_compounds[!my_compounds %in% unique(nonpolar_ri$query[!is.na(nonpolar_ri$RI)])]
failed_polar    <- my_compounds[!my_compounds %in% unique(polar_ri$query[!is.na(polar_ri$RI)])]
failed_either   <- unique(c(failed_nonpolar, failed_polar))

known_issues_result <- check_known_issues(failed_either, KNOWN_ISSUES_SHEET_URL)

if (!is.null(known_issues_result)) {
  write.csv(known_issues_result, "failed_compounds_lookup.csv", row.names = FALSE, fileEncoding = "UTF-8")
  cat("\n--- Failed compounds cross-checked against known-issues sheet ---\n")
  for (i in seq_len(nrow(known_issues_result))) {
    row <- known_issues_result[i, ]
    if (row$known_issue) {
      cat(sprintf("KNOWN ISSUE: %s - check manually: %s\n", row$compound, row$nist_url))
    } else {
      cat(sprintf("NEW failure (not yet documented): %s - consider adding to the known-issues sheet\n", row$compound))
    }
  }
} else if (length(failed_either) > 0) {
  cat("\n", length(failed_either), "compound(s) failed with no real RI data:",
      paste(failed_either, collapse = ", "), "\n")
  cat("(Set KNOWN_ISSUES_SHEET_URL to check these against a shared pointer list.)\n")
}

nonpolar_phase_summary  <- summarize_by(nonpolar_ri, "phase",  n_compounds)
polar_phase_summary     <- summarize_by(polar_ri,    "phase",  n_compounds)
nonpolar_family_summary <- summarize_by(nonpolar_ri, "family", n_compounds)
polar_family_summary    <- summarize_by(polar_ri,    "family", n_compounds)

nonpolar_stats_by_phase  <- summarize_ri_stats(nonpolar_ri, "phase")
polar_stats_by_phase     <- summarize_ri_stats(polar_ri,    "phase")
nonpolar_stats_by_family <- summarize_ri_stats(nonpolar_ri, "family")
polar_stats_by_family    <- summarize_ri_stats(polar_ri,    "family")

# ---- 5. Save everything ----------------------------------------------
write.csv(nonpolar_ri,             "nist_ri_nonpolar_raw.csv",        row.names = FALSE, fileEncoding = "UTF-8")
write.csv(polar_ri,                "nist_ri_polar_raw.csv",           row.names = FALSE, fileEncoding = "UTF-8")
write.csv(nonpolar_phase_summary,  "coverage_by_phase_nonpolar.csv",  row.names = FALSE, fileEncoding = "UTF-8")
write.csv(polar_phase_summary,     "coverage_by_phase_polar.csv",     row.names = FALSE, fileEncoding = "UTF-8")
write.csv(nonpolar_family_summary, "coverage_by_family_nonpolar.csv", row.names = FALSE, fileEncoding = "UTF-8")
write.csv(polar_family_summary,    "coverage_by_family_polar.csv",    row.names = FALSE, fileEncoding = "UTF-8")
write.csv(nonpolar_stats_by_phase,  "ri_stats_by_phase_nonpolar.csv",  row.names = FALSE, fileEncoding = "UTF-8")
write.csv(polar_stats_by_phase,     "ri_stats_by_phase_polar.csv",     row.names = FALSE, fileEncoding = "UTF-8")
write.csv(nonpolar_stats_by_family, "ri_stats_by_family_nonpolar.csv", row.names = FALSE, fileEncoding = "UTF-8")
write.csv(polar_stats_by_family,    "ri_stats_by_family_polar.csv",    row.names = FALSE, fileEncoding = "UTF-8")

# ---- 5b. Combined Excel workbook (one download, all tables as tabs) ---
# Same data as the individual CSVs above, bundled into a single .xlsx so
# you don't have to download 8+ separate files every run.
workbook_sheets <- list(
  "Raw_NonPolar"          = nonpolar_ri,
  "Raw_Polar"             = polar_ri,
  "Coverage_Phase_NonPolar"  = nonpolar_phase_summary,
  "Coverage_Phase_Polar"     = polar_phase_summary,
  "Coverage_Family_NonPolar" = nonpolar_family_summary,
  "Coverage_Family_Polar"    = polar_family_summary,
  "Stats_Phase_NonPolar"   = nonpolar_stats_by_phase,
  "Stats_Phase_Polar"      = polar_stats_by_phase,
  "Stats_Family_NonPolar"  = nonpolar_stats_by_family,
  "Stats_Family_Polar"     = polar_stats_by_family
)
if (!is.null(known_issues_result)) {
  workbook_sheets[["Failed_Compounds"]] <- known_issues_result
}
write_xlsx(workbook_sheets, "gc_ri_coverage_results.xlsx")

# ---- 6. Console summary ------------------------------------------------
cat("\n================= SUMMARY =================\n")
cat("Total compounds queried:", n_compounds, "\n\n")
cat("Top non-polar phases:\n"); print(head(nonpolar_phase_summary, 5))
cat("\nTop polar phases:\n");   print(head(polar_phase_summary, 5))
cat("\nNon-polar coverage by family:\n"); print(nonpolar_family_summary)
cat("\nPolar coverage by family:\n");     print(polar_family_summary)
cat("\nFiles written to your working directory:\n")
cat(" - gc_ri_coverage_results.xlsx (EVERYTHING below, as separate tabs in one file - probably all you need to download)\n")
cat(" - nist_ri_nonpolar_raw.csv / nist_ri_polar_raw.csv (full literature records)\n")
cat(" - coverage_by_phase_nonpolar.csv / _polar.csv (per-exact-phase ranking)\n")
cat(" - coverage_by_family_nonpolar.csv / _polar.csv (per-family ranking)\n")
cat(" - ri_stats_by_phase_nonpolar.csv / _polar.csv (mean/SD RI per compound+phase+type)\n")
cat(" - ri_stats_by_family_nonpolar.csv / _polar.csv (mean/SD RI per compound+family+type)\n")
cat(" - failed_compounds_lookup.csv (if any compounds failed and KNOWN_ISSUES_SHEET_URL was set)\n")
cat("Any phase family showing UNCLASSIFIED wasn't in the lookup table -\n")
cat("look it up manually and add it to phase_family_table above.\n")


### Getting your results

**The easiest option: `gc_ri_coverage_results.xlsx`** - every table below as a separate tab in one workbook, so you don't have to download 8+ separate files each run.

The same data is also written as individual CSV files into Colab's temporary storage, if you need just one table on its own:
- `nist_ri_nonpolar_raw.csv` / `nist_ri_polar_raw.csv` - full literature RI records, tagged by RI type (`ri_type` column)
- `coverage_by_phase_nonpolar.csv` / `_polar.csv` - ranking of exact GC phases
- `coverage_by_family_nonpolar.csv` / `_polar.csv` - ranking of phase families
- `ri_stats_by_phase_nonpolar.csv` / `_polar.csv` - count/mean/SD/min/max RI per compound + phase + RI type
- `ri_stats_by_family_nonpolar.csv` / `_polar.csv` - the same, rolled up to family level
- `failed_compounds_lookup.csv` - any compounds with no RI data, cross-checked against the known-issues sheet

**To download:** click the folder icon in the left sidebar, find the file(s), right-click, and choose **Download**. Colab's storage is temporary and is wiped when the session ends, so download anything you want to keep.

The console output above also prints a live summary so you can get a quick read without downloading anything - including which method (CAS or name) succeeded for each compound.

### If you see "UNCLASSIFIED - add to lookup table"
The phase-family lookup table is curated but not exhaustive. If a phase you get back isn't recognized, look it up (check the manufacturer's datasheet for its underlying chemistry - don't assume from the name alone; several phases look like they belong to one family but are actually another, e.g. CP-Sil 5 CB vs CP-Sil 8 CB) and add a row to `phase_family_table` in the code above. As of this version, **LM-120, CBP-1, CBP-5, MS5, and SBP-5** are known unresolved phases - documentation for their chemistry could not be found, so they're intentionally left unclassified rather than guessed.

### If you see "NEW failure (not yet documented)"
This means a compound failed and isn't yet in the known-issues sheet. If you manually check it on NIST WebBook and confirm it's a genuine known-issue case (not just a typo in the compound name), submit it via the Google Form for the moderator to review and add.